<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/registrasion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HÜCRE 1: ANTsPy Kurulumu
# ============================================================
!pip install antspyx -q

In [ ]:
# ============================================================
# HÜCRE 2: GPU Kontrolü
# ============================================================
import torch

if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print("DİKKAT: GPU bulunamadı!")

GPU Aktif: NVIDIA A100-SXM4-40GB


In [ ]:
# ============================================================
# HÜCRE 3: Google Drive Bağlantısı
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# ============================================================
# HÜCRE 4: Konfigürasyon
# ============================================================
import os



CONFIG_CN = {
    "kategori"    : "CN",
    "drive_kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_striping/skull_striping_CN',
    "drive_hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasion/Registrasion_CN'
}


In [ ]:


CONFIG_EMCI = {
    "kategori"    : "EMCI",
    "drive_kaynak": '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_striping/skull_striping_EMCI',
    "drive_hedef" : '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Registrasion/Registrasion_EMCI'
}


In [ ]:
# ============================================================
# HÜCRE 5: MNI Şablonunu Yükle
# ============================================================
import ants

mni_sablon = ants.get_ants_data('mni')
mni_img    = ants.image_read(mni_sablon)

print(f"MNI Şablonu : {mni_sablon}")
print(f"Boyut       : {mni_img.shape}")
print(f"Spacing     : {mni_img.spacing}")

MNI Şablonu : /root/.antspy/mni.nii.gz
Boyut       : (182, 218, 182)
Spacing     : (1.0, 1.0, 1.0)


In [ ]:
# ============================================================
# HÜCRE 6: FONKSİYON TANIMI (Düzeltilmiş)
# ============================================================
import os
import shutil
import ants

def mni_kaydet(config, mni_img):
    drive_kaynak = config["drive_kaynak"]
    drive_hedef = config["drive_hedef"]
    etiket = config["kategori"]

    local_input = f'/content/local_mni_input_{etiket}'
    local_output = f'/content/local_mni_output_{etiket}'

    os.makedirs(drive_hedef, exist_ok=True)
    os.makedirs(local_input, exist_ok=True)
    os.makedirs(local_output, exist_ok=True)

    dosyalar = sorted([
        f for f in os.listdir(drive_kaynak)
        if f.endswith('.nii.gz') and '_bet' not in f
    ])
    toplam = len(dosyalar)
    basarili = 0
    atlanan = 0
    hatali = 0

    print(f"\n {etiket} MNI Registration başlıyor ({toplam} dosya)\n")

    for i, ad in enumerate(dosyalar, 1):
        drive_cikti = os.path.join(drive_hedef, ad)
        local_girdi = os.path.join(local_input, ad)
        local_cikti = os.path.join(local_output, ad)
        matris_prefix = os.path.join(local_output, ad.replace('.nii.gz', '_transform'))

        if os.path.exists(drive_cikti):
            print(f"  [{i}/{toplam}] Atlandı: {ad}")
            atlanan += 1
            continue

        print(f" [{i}/{toplam}] Hizalanıyor: {ad}")

        try:
            shutil.copy(os.path.join(drive_kaynak, ad), local_girdi)

            hasta_img = ants.image_read(local_girdi)

            kayit = ants.registration(
                fixed=mni_img,
                moving=hasta_img,
                type_of_transform='antsRegistrationSyNQuick[s]',
                outprefix=matris_prefix
            )

            ants.image_write(kayit['warpedmovout'], local_cikti)

            shutil.copy(local_cikti, drive_cikti)

            print(f" Başarılı: {ad}")
            basarili += 1

        except Exception as e:
            print(f" Hata ({ad}): {str(e)}")
            hatali += 1

        finally:
            for klasor in [local_input, local_output]:
                for f in os.listdir(klasor):
                    os.remove(os.path.join(klasor, f))

    print(f"\n {etiket} MNI Registration Özeti:")
    print(f"   Toplam   : {toplam}")
    print(f"   Başarılı : {basarili}")
    print(f"   Atlanan  : {atlanan}")
    print(f"   Hatalı   : {hatali}")
    print(f"    {etiket} grubu tamamlandı.")

In [ ]:
# Desteklenen transform tiplerini gör
import ants
help(ants.registration)

Help on function registration in module ants.registration.registration:

registration(fixed, moving, type_of_transform='SyN', initial_transform=None, outprefix='', mask=None, moving_mask=None, mask_all_stages=False, grad_step=0.2, flow_sigma=3, total_sigma=0, aff_metric='mattes', aff_sampling=32, aff_random_sampling_rate=0.2, syn_metric='mattes', syn_sampling=32, reg_iterations=(40, 20, 0), aff_iterations=(2100, 1200, 1200, 10), aff_shrink_factors=(6, 4, 2, 1), aff_smoothing_sigmas=(3, 2, 1, 0), write_composite_transform=False, verbose=False, multivariate_extras=None, restrict_transformation=None, smoothing_in_mm=False, singleprecision=True, use_legacy_histogram_matching=False, **kwargs)
    Register a pair of images either through the full or simplified
    interface to the ANTs registration method.

    ANTsR function: `antsRegistration`

    Arguments
    ---------
    fixed : ANTsImage
        fixed image to which we register the moving image.

    moving : ANTsImage
        moving

In [ ]:
import ants
import shutil
import os

# Tek bir dosyayla test
test_kaynak = '/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/skull_striping/skull_striping_CN/002_S_0559.nii.gz'
test_lokal  = '/content/test_dosya.nii.gz'

# Kopyala
shutil.copy(test_kaynak, test_lokal)
print(f"Dosya boyutu: {os.path.getsize(test_lokal)} bytes")

# Oku
img = ants.image_read(test_lokal)
print(f"Boyut   : {img.shape}")
print(f"Spacing : {img.spacing}")
print("✅ Okuma başarılı")

Dosya boyutu: 4345658 bytes
Boyut   : (256, 256, 170)
Spacing : (1.0, 1.0, 1.2000000476837158)
✅ Okuma başarılı


In [ ]:
# ============================================================
# HÜCRE 7: CN Grubunu Çalıştır
# ============================================================
# ============================================================
# HÜCRE 7: CN Grubunu Çalıştır
# ============================================================
mni_kaydet(CONFIG_CN, mni_img)


 CN MNI Registration başlıyor (30 dosya)

 [1/30] Hizalanıyor: 002_S_0559.nii.gz
 Başarılı: 002_S_0559.nii.gz
 [2/30] Hizalanıyor: 010_S_0420.nii.gz
 Başarılı: 010_S_0420.nii.gz
 [3/30] Hizalanıyor: 011_S_0002.nii.gz
 Başarılı: 011_S_0002.nii.gz
 [4/30] Hizalanıyor: 011_S_0016.nii.gz
 Başarılı: 011_S_0016.nii.gz
 [5/30] Hizalanıyor: 011_S_0021.nii.gz
 Başarılı: 011_S_0021.nii.gz
 [6/30] Hizalanıyor: 011_S_0022.nii.gz
 Başarılı: 011_S_0022.nii.gz
 [7/30] Hizalanıyor: 012_S_1133.nii.gz
 Başarılı: 012_S_1133.nii.gz
 [8/30] Hizalanıyor: 018_S_0043.nii.gz
 Başarılı: 018_S_0043.nii.gz
 [9/30] Hizalanıyor: 018_S_0055.nii.gz
 Başarılı: 018_S_0055.nii.gz
 [10/30] Hizalanıyor: 018_S_0369.nii.gz
 Başarılı: 018_S_0369.nii.gz
 [11/30] Hizalanıyor: 023_S_0061.nii.gz
 Başarılı: 023_S_0061.nii.gz
 [12/30] Hizalanıyor: 027_S_0403.nii.gz
 Başarılı: 027_S_0403.nii.gz
 [13/30] Hizalanıyor: 032_S_0479.nii.gz
 Başarılı: 032_S_0479.nii.gz
 [14/30] Hizalanıyor: 032_S_0677.nii.gz
 Başarılı: 032_S_0677.nii.gz


In [ ]:
# ============================================================
# HÜCRE 8: EMCI Grubunu Çalıştır
# ============================================================
# ============================================================
# HÜCRE 7: CN Grubunu Çalıştır
# ============================================================
mni_kaydet(CONFIG_EMCI, mni_img)


 EMCI MNI Registration başlıyor (30 dosya)

 [1/30] Hizalanıyor: 002_S_2010.nii.gz
 Başarılı: 002_S_2010.nii.gz
 [2/30] Hizalanıyor: 009_S_2208.nii.gz
 Başarılı: 009_S_2208.nii.gz
 [3/30] Hizalanıyor: 012_S_4128.nii.gz
 Başarılı: 012_S_4128.nii.gz
 [4/30] Hizalanıyor: 018_S_2155.nii.gz
 Başarılı: 018_S_2155.nii.gz
 [5/30] Hizalanıyor: 022_S_2167.nii.gz
 Başarılı: 022_S_2167.nii.gz
 [6/30] Hizalanıyor: 032_S_2240.nii.gz
 Başarılı: 032_S_2240.nii.gz
 [7/30] Hizalanıyor: 036_S_2380.nii.gz
 Başarılı: 036_S_2380.nii.gz
 [8/30] Hizalanıyor: 057_S_2398.nii.gz
 Başarılı: 057_S_2398.nii.gz
 [9/30] Hizalanıyor: 068_S_2316.nii.gz
 Başarılı: 068_S_2316.nii.gz
 [10/30] Hizalanıyor: 068_S_4067.nii.gz
 Başarılı: 068_S_4067.nii.gz
 [11/30] Hizalanıyor: 072_S_2037.nii.gz
 Başarılı: 072_S_2037.nii.gz
 [12/30] Hizalanıyor: 072_S_2072.nii.gz
 Başarılı: 072_S_2072.nii.gz
 [13/30] Hizalanıyor: 072_S_2083.nii.gz
 Başarılı: 072_S_2083.nii.gz
 [14/30] Hizalanıyor: 072_S_2093.nii.gz
 Başarılı: 072_S_2093.nii.g

In [ ]:
# ============================================================
# HÜCRE 9: Sonuç Doğrulama
# ============================================================
def mni_kontrol(config):
    hedef  = config["drive_hedef"]
    etiket = config["kategori"]

    dosyalar   = os.listdir(hedef)
    goruntuler = sorted([f for f in dosyalar if f.endswith('.nii.gz')])
    matrisler  = [f for f in dosyalar if '_transform' in f]

    print(f"📁 {etiket} MNI Klasörü:")
    print(f"   Hizalanmış görüntü  : {len(goruntuler)}")
    print(f"   Dönüşüm dosyaları   : {len(matrisler)}")

    eksik = [
        f for f in goruntuler
        if not any(f.replace('.nii.gz', '_transform') in m for m in matrisler)
    ]
    if eksik:
        print(f"   ⚠️  Dönüşüm dosyası eksik: {eksik}")
    else:
        print(f"   ✅ Tüm görüntülerin dönüşüm dosyası mevcut.")

mni_kontrol(CONFIG_CN)
print()
mni_kontrol(CONFIG_EMCI)

📁 CN MNI Klasörü:
   Hizalanmış görüntü  : 30
   Dönüşüm dosyaları   : 0
   ⚠️  Dönüşüm dosyası eksik: ['002_S_0559.nii.gz', '010_S_0420.nii.gz', '011_S_0002.nii.gz', '011_S_0016.nii.gz', '011_S_0021.nii.gz', '011_S_0022.nii.gz', '012_S_1133.nii.gz', '018_S_0043.nii.gz', '018_S_0055.nii.gz', '018_S_0369.nii.gz', '023_S_0061.nii.gz', '027_S_0403.nii.gz', '032_S_0479.nii.gz', '032_S_0677.nii.gz', '036_S_1023.nii.gz', '037_S_0303.nii.gz', '057_S_0818.nii.gz', '067_S_0059.nii.gz', '073_S_0386.nii.gz', '082_S_0304.nii.gz', '099_S_0040.nii.gz', '099_S_0090.nii.gz', '099_S_0533.nii.gz', '109_S_0876.nii.gz', '114_S_0166.nii.gz', '114_S_0173.nii.gz', '116_S_0382.nii.gz', '141_S_0726.nii.gz', '941_S_1197.nii.gz', '941_S_1202.nii.gz']

📁 EMCI MNI Klasörü:
   Hizalanmış görüntü  : 30
   Dönüşüm dosyaları   : 0
   ⚠️  Dönüşüm dosyası eksik: ['002_S_2010.nii.gz', '009_S_2208.nii.gz', '012_S_4128.nii.gz', '018_S_2155.nii.gz', '022_S_2167.nii.gz', '032_S_2240.nii.gz', '036_S_2380.nii.gz', '057_S_2398.